# LNCLIP-DF — Dual-Input Deepfake Detection

**Architecture:** CLIP ViT-L/14 + LN-tuning + Dual-Input (Face + Full Frame) + Angular Margin

**Dataset:** 1,056 videos from 3 sources (Drive: `MyDrive/VIPER/`)

**Runtime:** ~2.5h preprocessing + ~40min training on T4

In [ ]:
# ═══ CELL 1: Setup ═══════════════════════════════════════════
import torch
assert torch.cuda.is_available(), 'Need T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
!pip install -q open_clip_torch insightface onnxruntime-gpu opencv-python-headless scipy scikit-learn tqdm pandas matplotlib
print('✓ Ready')

In [ ]:
# ═══ CELL 2: Mount + Clone ═══════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob
from pathlib import Path

if not os.path.exists('/content/eidon-deepfake'):
    !git clone https://github.com/rxbinsingh/eidon-deepfake /content/eidon-deepfake
else:
    !git -C /content/eidon-deepfake pull --quiet

os.chdir('/content/eidon-deepfake')
sys.path.insert(0, '/content/eidon-deepfake')

BASE = '/content/drive/MyDrive/VIPER'
DATA_DIRS = [f'{BASE}/dataset_production', f'{BASE}/dataset_week2', f'{BASE}/dfd_dataset']
CACHE_DIR = f'{BASE}/cache_dual'
SAVE_DIR  = f'{BASE}/checkpoints_dual'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

import pandas as pd
total = 0
for d in DATA_DIRS:
    if os.path.exists(f'{d}/metadata.csv'):
        n = len(pd.read_csv(f'{d}/metadata.csv'))
        total += n
        print(f'  ✓ {Path(d).name}: {n} videos')
    else:
        print(f'  ✗ {Path(d).name}: NOT FOUND')
print(f'Total: {total}')

In [ ]:
# ═══ CELL 3: Find all videos ════════════════════════════════
import pandas as pd
import numpy as np

def get_all_videos():
    samples = []
    for d in DATA_DIRS:
        meta_path = f'{d}/metadata.csv'
        if not os.path.exists(meta_path): continue
        meta = pd.read_csv(meta_path)
        found, missed = 0, 0
        for _, row in meta.iterrows():
            label_str = row.get('label', 'unknown')
            label = 0 if label_str == 'real' else 1
            cat = row.get('category', label_str)
            fname = row['filename']
            possible = [
                f'{d}/{cat}/{fname}', f'{d}/{label_str}/{fname}',
            ]
            if 'original_path' in row and pd.notna(row['original_path']):
                possible.append(f'{d}/{row["original_path"]}')
                possible.append(f'{d}/{Path(row["original_path"]).parent}/{fname}')
            if 'dfd' in d.lower() or 'DFD' in d:
                possible += [f'{d}/DFD_original sequences/{fname}',
                             f'{d}/DFD_manipulated_sequences/{fname}',
                             f'{d}/DFD_manipulated_sequences/DFD_manipulated_sequences/{fname}']
            for vp in possible:
                if os.path.exists(vp):
                    samples.append((vp, label, cat)); found += 1; break
            else:
                missed += 1
        print(f'  {Path(d).name}: found={found}, missed={missed}')
    return samples

all_videos = get_all_videos()
print(f'\nTotal: {len(all_videos)} (Real: {sum(1 for _,l,_ in all_videos if l==0)}, Fake: {sum(1 for _,l,_ in all_videos if l==1)})')

In [ ]:
# ═══ CELL 4: Preprocess (dual: face crops + full frames) ════
# Cached to Drive. Safe to restart.
import cv2
from tqdm import tqdm
from src.preprocessing import preprocess_video

# Patch BCR (mediapipe broken)
import src.preprocessing as prep_mod
try:
    import src.anchor_extractor as ae
    ae.build_biomech_anchor = lambda frames: None
except: pass

already = len(glob.glob(f'{CACHE_DIR}/*.npz'))
print(f'Already cached: {already} | To process: {len(all_videos)-already}')

ok, fail = 0, 0
for vp, label, cat in tqdm(all_videos, desc='Preprocessing'):
    cp = f'{CACHE_DIR}/{Path(vp).stem}.npz'
    if os.path.exists(cp): ok += 1; continue
    try:
        p = preprocess_video(vp, num_frames=16, n_anchor=8)
        if not p['valid']: fail += 1; continue
        # Save both face crops and full frames
        face_crops = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['face_crops']] if p['face_valid'] else []
        full_frames = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['full_frames']]
        save_data = {
            'full_frames': np.stack(full_frames),
            'label': np.array(label),
            'face_valid': np.array(p['face_valid']),
        }
        if face_crops:
            save_data['face_crops'] = np.stack(face_crops)
        np.savez_compressed(cp, **save_data)
        ok += 1
    except Exception as e:
        fail += 1

total = ok + fail
print(f'\nDone. OK: {ok} | Failed: {fail} | Rate: {ok/max(total,1)*100:.0f}%')

In [ ]:
# ═══ CELL 5: Build Model + Datasets ══════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F, random
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from PIL import Image
from src.lnclip_model import build_lnclip_model

device = 'cuda'
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# Compression augmentation
class CompressAug:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        arr = np.array(img)
        if random.random() < 0.5:
            q = random.randint(30, 95)
            _, enc = cv2.imencode('.jpg', cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
            arr = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if random.random() < 0.3:
            s = random.uniform(0.5, 2.0); k = int(s*4)|1
            arr = cv2.GaussianBlur(arr, (k,k), s)
        if random.random() < 0.3:
            sc = random.uniform(0.5, 0.9); h,w = arr.shape[:2]
            arr = cv2.resize(cv2.resize(arr, (int(w*sc),int(h*sc))), (w,h))
        return Image.fromarray(arr)

CLIP_TF = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.48145466,0.4578275,0.40821073],[0.26862954,0.26130258,0.27577711])])

class DualDataset(Dataset):
    def __init__(self, samples, cache_dir, train=True):
        self.samples = [(p,l) for p,l,_ in samples if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir = cache_dir
        self.aug = CompressAug(0.5) if train else None
        self.flip = T.RandomHorizontalFlip(0.5) if train else T.RandomHorizontalFlip(0)
        print(f'  Dataset: {len(self.samples)} ({"train" if train else "eval"})')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        vp, label = self.samples[idx]
        data = np.load(f'{self.cache_dir}/{Path(vp).stem}.npz')
        full_frames = data['full_frames']  # (T, 224, 224, 3)
        face_valid = bool(data['face_valid'])
        face_crops = data.get('face_crops', None)  # (T, 224, 224, 3) or None

        def process_crops(crops_arr, n=16):
            T_actual = min(len(crops_arr), n)
            tensors = []
            for i in range(T_actual):
                img = Image.fromarray(crops_arr[i])
                if self.aug: img = self.aug(img)
                img = self.flip(img)
                tensors.append(CLIP_TF(img))
            while len(tensors) < n: tensors.append(tensors[-1])
            return torch.stack(tensors[:n])

        # Full frames (always available)
        full_t = process_crops(full_frames)

        # Face crops (zeros if not available)
        if face_valid and face_crops is not None and len(face_crops) >= 4:
            face_t = process_crops(face_crops)
        else:
            face_t = torch.zeros_like(full_t)

        return {'face_crops': face_t, 'full_frames': full_t,
                'label': torch.tensor(label, dtype=torch.long)}

# Split
shuffled = all_videos.copy()
random.shuffle(shuffled)
n = len(shuffled)
train_s, val_s, test_s = shuffled[:int(.7*n)], shuffled[int(.7*n):int(.8*n)], shuffled[int(.8*n):]
print(f'Split: Train={len(train_s)} Val={len(val_s)} Test={len(test_s)}')

train_ds = DualDataset(train_s, CACHE_DIR, train=True)
val_ds   = DualDataset(val_s, CACHE_DIR, train=False)
test_ds  = DualDataset(test_s, CACHE_DIR, train=False)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# Model
model, _ = build_lnclip_model(device=device)
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params')

In [ ]:
# ═══ CELL 6: Training ════════════════════════════════════════
EPOCHS = 20
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

def train_epoch():
    model.train()
    all_l, all_p, loss_sum = [], [], 0.0
    for batch in tqdm(train_loader, leave=False):
        face = batch['face_crops'].to(device)
        full = batch['full_frames'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, _ = model(face, full, labels)
            loss = F.cross_entropy(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()
        all_l.extend(labels.cpu().numpy())
        all_p.extend(F.softmax(logits.detach(), dim=-1)[:,1].cpu().numpy())
    return loss_sum/len(train_loader), roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    all_l, all_p = [], []
    for batch in tqdm(loader, leave=False):
        face = batch['face_crops'].to(device)
        full = batch['full_frames'].to(device)
        with torch.amp.autocast('cuda'):
            logits, _ = model(face, full, labels=None)
        all_l.extend(batch['label'].numpy())
        all_p.extend(F.softmax(logits, dim=-1)[:,1].cpu().numpy())
    auc = roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0
    return auc, accuracy_score(all_l, [1 if p>.5 else 0 for p in all_p]), all_l, all_p

best_auc, patience_ct = 0.0, 0
print(f'Training for up to {EPOCHS} epochs...\n')
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_auc = train_epoch()
    vl_auc, vl_acc, _, _ = eval_epoch(val_loader)
    scheduler.step()
    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc; patience_ct = 0
        torch.save(model.state_dict(), f'{SAVE_DIR}/best.pt')
        flag = '  ← best'
    else: patience_ct += 1
    print(f'Epoch {epoch:02d}/{EPOCHS}  Loss={tr_loss:.4f}  Train={tr_auc:.4f}  Val={vl_auc:.4f}  Acc={vl_acc:.3f}{flag}')
    if patience_ct >= 7: print('Early stop'); break
print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
# ═══ CELL 7: Test + TTA ══════════════════════════════════════
model.load_state_dict(torch.load(f'{SAVE_DIR}/best.pt', map_location=device))
model.eval()
all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test+TTA'):
        face = batch['face_crops'].to(device)
        full = batch['full_frames'].to(device)
        with torch.amp.autocast('cuda'):
            l1, _ = model(face, full, labels=None)
            l2, _ = model(torch.flip(face,[-1]), torch.flip(full,[-1]), labels=None)
        probs = (F.softmax(l1,dim=-1)[:,1] + F.softmax(l2,dim=-1)[:,1]) / 2
        all_labels.extend(batch['label'].numpy())
        all_probs.extend(probs.cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
preds = [1 if p>.5 else 0 for p in all_probs]
acc = accuracy_score(all_labels, preds)
cm = confusion_matrix(all_labels, preds)
print(f'\n{"═"*50}')
print(f'  FINAL TEST RESULTS')
print(f'{"═"*50}')
print(f'  AUC:      {auc:.4f}')
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'  TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')
print(f'\n{classification_report(all_labels, preds, target_names=["Real","Fake"])}')

In [ ]:
# ═══ CELL 8: Per-Category + Save ═════════════════════════════
import json
cat_l, cat_p = {}, {}
for (vp,label,cat), prob in zip(test_s, all_probs[:len(test_s)]):
    if cat not in cat_l: cat_l[cat], cat_p[cat] = [], []
    cat_l[cat].append(label); cat_p[cat].append(prob)

print(f'{"Category":<25} {"AUC":>8} {"N":>5}')
print('─'*40)
real_l, real_p = cat_l.get('real',[]), cat_p.get('real',[])
per_type = {}
for cat in sorted(cat_l.keys()):
    if cat == 'real': continue
    cl = real_l + cat_l[cat]; cp = real_p + cat_p[cat]
    if len(set(cl)) < 2: continue
    a = roc_auc_score(cl, cp)
    per_type[cat] = round(a, 4)
    print(f'{cat:<25} {a:>8.4f} {len(cat_l[cat]):>5}')
print('─'*40)
print(f'{"ALL":<25} {auc:>8.4f} {len(all_labels):>5}')

results = {'model': 'LNCLIP-DF Dual-Input', 'test_auc': round(auc,4),
           'accuracy': round(acc,4), 'best_val_auc': round(best_auc,4),
           'per_type_auc': per_type, 'cm': cm.tolist()}
with open(f'{SAVE_DIR}/results.json','w') as f: json.dump(results, f, indent=2)
print(f'\nSaved to {SAVE_DIR}/')